# Resume SignJoey training (GPU) from an `lsu-pria` run

This notebook resumes **backend (SignJoey)** training using a previous `lsu-pria` run folder.

It is designed for runs like:
- `whisperx_slt_mps_nightly_20260531_0052`

## What you need on Google Drive

Copy your run folder into Drive, e.g.:
- `MyDrive/lsu-pria-runs/whisperx_slt_mps_nightly_20260531_0052/`

The run must contain at least:
- `export/backend/neccam_slt/data/` (backend bundle)
- `train/external_backend_config.yaml` (generated config)
- `train/external_backend_model/*.ckpt` (checkpoint to resume from)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ====== EDIT THESE ======
RUN_NAME = "whisperx_slt_mps_nightly_20260531_0052"
DRIVE_RUNS_DIR = "/content/drive/MyDrive/lsu-pria-runs"  # where you copied the run folder

# Optional: start a NEW model_dir for the resumed training
NEW_TRAIN_TAG = "colab_resume_gpu"

# If you want to train longer, set new total epochs here
# (SignJoey uses the value in config as the *target total epochs*.)
NEW_TOTAL_EPOCHS = 30

# Batch size to use on GPU (adjust if you hit OOM)
NEW_BATCH_SIZE = 8
assert NEW_BATCH_SIZE >= 1


In [ ]:
import os
from pathlib import Path

run_dir = Path(DRIVE_RUNS_DIR) / RUN_NAME
assert run_dir.exists(), f"Run folder not found: {run_dir}"

export_dir = run_dir / "export"
train_dir = run_dir / "train"
cfg_in = train_dir / "external_backend_config.yaml"
model_dir_old = train_dir / "external_backend_model"

assert (export_dir / "backend" / "neccam_slt" / "data").exists(), "Missing backend bundle under export/backend/neccam_slt/data"
assert cfg_in.exists(), f"Missing config: {cfg_in}"
assert model_dir_old.exists(), f"Missing model dir: {model_dir_old}"

print("run_dir:", run_dir)
print("cfg_in:", cfg_in)
print("model_dir_old:", model_dir_old)
print("ckpts:", sorted(p.name for p in model_dir_old.glob("*.ckpt"))[:10])


## Install dependencies + SignJoey (GPU)

This installs PyTorch CUDA for Colab and clones the backend repo.

If you use a different fork than `slt-mps`, change the git URL below.


In [ ]:
!nvidia-smi -L || true
!python -V

# PyTorch (CUDA)
!pip -q install --upgrade pip
!pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Common deps needed by SignJoey + metrics
!pip -q install pyyaml portalocker tensorboard sacrebleu numpy

# Clone backend repo (SignJoey fork)
!rm -rf /content/slt-backend
!git clone --depth 1 https://github.com/sbarcelona11/slt.git /content/slt-backend
!pip -q install -e /content/slt-backend

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


## Create a patched config for Colab resume

We **do not** resume into the same `model_dir`, because SignJoey's `overwrite: false` would error if the directory already exists.

Instead we:
- create a new model dir under the run (Drive)
- point `load_model` to your last/best checkpoint
- set `use_cuda: true`
- set new `epochs` and `batch_size`

You can change which checkpoint to resume from below.


In [ ]:
import yaml

cfg = yaml.safe_load(cfg_in.read_text())

backend_data_path = export_dir / "backend" / "neccam_slt" / "data"
cfg["data"]["data_path"] = str(backend_data_path)

# Pick checkpoint: prefer best.ckpt symlink if present, else max-number ckpt
best_ckpt = model_dir_old / "best.ckpt"
if best_ckpt.exists():
    ckpt_path = best_ckpt
else:
    ckpts = list(model_dir_old.glob("*.ckpt"))
    assert ckpts, f"No ckpt found in {model_dir_old}"
    # numeric sort by stem if possible
    def _key(p):
        try:
            return int(p.stem)
        except Exception:
            return -1
    ckpt_path = sorted(ckpts, key=_key)[-1]

new_model_dir = train_dir / f"external_backend_model_{NEW_TRAIN_TAG}"
cfg["training"]["model_dir"] = str(new_model_dir)

# Resume from checkpoint
cfg["training"]["load_model"] = str(ckpt_path)
cfg["training"]["reset_best_ckpt"] = False
cfg["training"]["reset_scheduler"] = False
cfg["training"]["reset_optimizer"] = False

# IMPORTANT: don't delete existing dirs
cfg["training"]["overwrite"] = False

# GPU training
cfg["training"]["use_cuda"] = True

# Keep your project constraint: recognition_loss_weight stays 0.0 unless you change it
# (Leave as-is)

# Extend training
cfg["training"]["epochs"] = int(NEW_TOTAL_EPOCHS)
cfg["training"]["batch_size"] = int(NEW_BATCH_SIZE)

cfg_out = train_dir / f"external_backend_config_{NEW_TRAIN_TAG}.yaml"
cfg_out.write_text(yaml.safe_dump(cfg, sort_keys=False))

print("Resuming from ckpt:", ckpt_path)
print("Writing patched cfg:", cfg_out)
print("New model_dir:", new_model_dir)


In [ ]:
# Quick sanity check: show key fields
print("data_path:", cfg["data"]["data_path"])
print("model_dir:", cfg["training"]["model_dir"])
print("load_model:", cfg["training"].get("load_model"))
print("overwrite:", cfg["training"].get("overwrite"))
print("use_cuda:", cfg["training"].get("use_cuda"))
print("epochs:", cfg["training"].get("epochs"))
print("batch_size:", cfg["training"].get("batch_size"))
print("recognition_loss_weight:", cfg["training"].get("recognition_loss_weight"))
print("translation_loss_weight:", cfg["training"].get("translation_loss_weight"))


## Resume training

This will write new logs under the new `model_dir` (Drive), including:
- `train.log`
- `validations.txt`
- `tensorboard/`

Tip: open a second cell with `tail -f .../train.log` while training runs.


In [ ]:
!python -m signjoey train "$cfg_out"

In [ ]:
# Watch logs (run this in another cell while training)
log_path = Path(cfg["training"]["model_dir"]) / "train.log"
val_path = Path(cfg["training"]["model_dir"]) / "validations.txt"
print("train.log:", log_path)
print("validations.txt:", val_path)
!tail -n 50 "$log_path" || true
!tail -n 20 "$val_path" || true


## TensorBoard

If you want TensorBoard inside Colab, run the cell below.


In [ ]:
%load_ext tensorboard
tb_dir = str(Path(cfg["training"]["model_dir"]) / "tensorboard")
print("TensorBoard dir:", tb_dir)
%tensorboard --logdir $tb_dir